# OBE solve-time speed investigation

This notebook benchmarks OBE **solve time only**. OBE-system construction and Rust plan preparation are done before the timed solve sections and are reported only as context.

## Setup

The benchmark helper writes CSV files, figures, and a Markdown report under `examples/lindblad/`. The default run is intentionally bounded so it can be rerun interactively.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

_here = Path.cwd()
for _root in [_here, *_here.parents]:
    if (_root / "centrex_tlf").exists():
        if str(_root) not in sys.path:
            sys.path.insert(0, str(_root))
        break

from examples.lindblad import obe_solve_speed_benchmark as bench

plt.rcParams.update({"font.size": 14})

## Run Benchmarks

The default benchmark uses a small scan grid and thread sweep. For a longer scaling run, use values such as `scan_points=(25, 101, 401)` and `thread_counts=(1, 2, 4, 8, 12, 16)`.

In [ ]:
# This cell regenerates all CSVs, figures, and the Markdown report.
# The sparse expm_multiply comparison is slow for this model; set
# run_exponential=False for quick iteration.
tables = bench.run_benchmarks(
    scan_points=(9,),
    thread_counts=(1, 2),
    run_exponential=True,
)

## Load Saved Results

Use this cell when the benchmark has already been run and you only want to inspect the latest saved results.

In [ ]:
tables = {
    path.stem: pd.read_csv(path)
    for path in sorted(bench.RESULTS_DIR.glob("*.csv"))
}
sorted(tables)

## System Sizes

In [ ]:
tables["system_summary"]

## Single-Trajectory Results

In [ ]:
tables["single_trajectory"]

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(bench.FIGURES_DIR / "obe_solve_single_trajectory.png")))

## Frequency-Scan Scaling

In [ ]:
tables["scan_thread_scaling"]

In [ ]:
display(Image(filename=str(bench.FIGURES_DIR / "obe_solve_scan_thread_scaling.png")))

## Detuning-Dependent Work

In [ ]:
tables["detuning_stats"]

In [ ]:
display(Image(filename=str(bench.FIGURES_DIR / "obe_solve_rhs_by_detuning.png")))

## Constant-Coefficient Exponential Test

This tests the constant-field approximation `d rho / dt = L rho` with an augmented Liouvillian for photon counts.

In [ ]:
tables["exponential_comparison"]

In [ ]:
display(Image(filename=str(bench.FIGURES_DIR / "obe_solve_exponential_comparison.png")))

## Report

The Markdown report is regenerated by `bench.run_benchmarks(...)` and can also be refreshed from saved tables with `bench.write_report(...)`.

In [ ]:
print(bench.REPORT_PATH)
print(bench.REPORT_PATH.read_text(encoding="utf-8")[:2000])